In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import re
from pathlib import Path
from datetime import date, timedelta
from natsort import natsort_keygen, natsorted, ns
from xlsxwriter.utility import xl_col_to_name
from datetime import datetime

import sys
sys.path.insert(0, str(Path("../src").resolve()))
import ado_reporting as cl
import daily_charge as dc
import work_items_report as wir
import data_validation as dv

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format


In [2]:
# ==== USER CONFIG ====
project = "MarketRMS"

start_date = date(2026, 3, 1)  # <- report generated since that date

# today = date.today()

today = "2026-04-01"
today = datetime.strptime(today, "%Y-%m-%d").date()

# --- Validate 'today' ---
today = dv.validate_today(today)

DAYS_BACK = (today - start_date).days + 1

# Paths
BASE_DIR = Path("..") / "data"

# Categories
leaf_types = ["Task", "Bug"]
removed_state = "Removed"
doneish_statuses = {"Code Review", "Ready for UAT", "Ready for Prod", "In UAT", "Closed"}
daily_charge_output_path = BASE_DIR / f"{project}_daily_charge_report.xlsx"
today_ts = pd.Timestamp(today)
today_str = today.strftime("%Y-%m-%d")
output_file = BASE_DIR / f"{project}_ProjectStatus_{today_str}_ADO_API.xlsx"
output_file_2 = BASE_DIR / f"{project}_Utilisation_{today_str}.xlsx"
burndown_file = BASE_DIR / f"{project}_Burndown_{today_str}.xlsx"


# Prepare shapshots dataset for analysis

In [3]:
wi_raw_snapshots = cl.get_wi_snapshots(
    project=project,
    days_back=DAYS_BACK,
    removed_state=removed_state,
)

# --- Validate source data freshness ---
_latest_snap = pd.Timestamp(wi_raw_snapshots["SnapshotDate"].max()).date()
if _latest_snap < today - timedelta(days=1):
    raise RuntimeError(
        "Project snapshots data is outdated. Please run synchronisation with ADO."
    )


Historical data selected: 2026-02-28 - 2026-03-31
Latest live snapshot file: MarketRMS_live_snapshot_2026-04-01.parquet (date=2026-04-01)
Live rows loaded: 1923
Final dataset date range: 2026-02-28 → 2026-04-01


# Identify related sprints

In [4]:
sprints, current_sprint, previous_sprint = cl.get_sprint_info(wi_raw_snapshots, today)

previous_sprint, current_sprint

('Sprint 35 (16.03-27.03)', 'Sprint 36 (30.03-10.04)')

# Charge by day report

In [ ]:
# Compute daily charge tables (per assignee, per task) from snapshots.
# See src/daily_charge.py for the calculation.


In [ ]:
# --- Daily charge by Assignee ---
daily_charge_by_person = dc.compute_daily_charge_by_person(wi_raw_snapshots)


In [ ]:
# --- Daily charge by Task ---
daily_charge_by_task = dc.compute_daily_charge_by_task(wi_raw_snapshots)


In [ ]:
# --- Download Daily charge for Power BI ---
daily_charge_by_person_bi = dc.export_daily_charge_for_powerbi(
    daily_charge_by_person,
    BASE_DIR / f"{project}_PowerBI_daily_charge.csv",
)


# Report for Daily (Stand-Ups)

In [ ]:
# Build the daily meeting (stand-up) report.
# See src/work_items_report.py for the calculation.
daily_meeting_filtered, completed_cols = wir.compute_daily_meeting(
    wi_raw_snapshots, daily_charge_by_task, today
)
col_completed_earliest, col_completed_mid, col_completed_latest = completed_cols


# Sprint Burndown

In [14]:
# --- Burndown calculation function ---
def sprint_burndown(df_leaf, sprint_name):
    """
    Return DataFrame with columns: Date, Remaining Work
    for snapshots within the selected sprint window:
      IterationStart <= Snapshot Date <= (IterationEnd - 1 day).
    Assumes df_leaf has columns:
      - Sprint (IterationName), Snapshot Date, State,
        Remaining Work, IterationStart, IterationEnd.
    """
    if not sprint_name:
        return pd.DataFrame(columns=["Date", "RemainingWork"])
    
    sub = df_leaf.loc[df_leaf["IterationName"] == sprint_name].copy()
    
    if sub.empty:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # Resolve sprint dates from the rows
    sprint_start = sub["IterationStart"]
    sprint_end = sub["IterationEnd"]

    if sprint_start.isna().all():
        return pd.DataFrame(columns=["Date", "RemainingWork"])
    if sprint_end.isna().all():
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # include up to End - 1 day
    sprint_last_inclusive = sprint_end - pd.Timedelta(days=1)

    # Keep snapshots only within sprint window
    sub = sub.loc[
        (sub["SnapshotDate"] >= sprint_start) &
        (sub["SnapshotDate"] <= sprint_last_inclusive)
    ].copy()
    if sub.empty:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # Exclude removed items from burndown
    sub = sub[sub["State"] != removed_state]

    # Clear remaining work for items in done-ish states
    sub["RemainingBD"] = sub["RemainingWork"]
    sub.loc[sub["State"].isin(doneish_statuses), "RemainingBD"] = 0.0

    bd = (
        sub.groupby(sub["SnapshotDate"].dt.normalize())["RemainingBD"]
           .sum()
           .sort_index()
           .reset_index(name="Date")
           .rename(columns={"RemainingBD": "RemainingWork"})
    )
    return bd

In [15]:
# --- 1. Calculate BD for current and previous Iterations ---
bd_current = sprint_burndown(wi_raw_snapshots, current_sprint)
bd_prev = sprint_burndown(wi_raw_snapshots, previous_sprint)

In [16]:
# # --- 2. Calculate BD by assignee for current Iteration ---
leaf_curr = wi_raw_snapshots[
    (wi_raw_snapshots["IterationName"] == current_sprint)
    & (wi_raw_snapshots["SnapshotDate"].isin(bd_current.SnapshotDate.unique()))
].copy()

# Exclude 'Removed' from burndown
leaf_curr = leaf_curr[leaf_curr["State"] != removed_state]

# Apply your burndown rules
leaf_curr["RemainingBD"] = leaf_curr["RemainingWork"]
leaf_curr.loc[leaf_curr["State"].isin(doneish_statuses), "RemainingBD"] = 0.0

# Group by Snapshot Date + Assignee
bd_assignee = (
    leaf_curr.groupby(["SnapshotDate", "AssignedTo"])["RemainingBD"]
             .sum()
             .reset_index()
)

# Pivot: one column per assignee
bd_assignee_pivot = (
    bd_assignee
    .pivot(index="SnapshotDate", columns="AssignedTo", values="RemainingBD")
    .fillna(0.0)
    .sort_index()
)

# Flatten index/columns for Excel
bd_assignee_pivot = bd_assignee_pivot.reset_index().rename(columns={"SnapshotDate": "Date"})
assignee_cols = [c for c in bd_assignee_pivot.columns if c != "Date"]

In [17]:
from xlsxwriter.utility import xl_col_to_name
import pandas as pd

# --- Prep summary (stand-up) table ---
summary = daily_meeting_filtered.copy()

# Strip timezone for Excel if present
for col in ["IterationStart", "IterationEnd"]:
    if col in summary.columns and pd.api.types.is_datetime64_any_dtype(summary[col]):
        try:
            summary[col] = summary[col].dt.tz_convert(None)
        except AttributeError:
            pass

# Columns to highlight in blue
completed_cols = [col_completed_earliest, col_completed_mid, col_completed_latest]

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    # --- Write sheets in desired order ---

    # 1) Status
    summary.to_excel(writer, index=False, sheet_name="Status")

    # 2) CurrentByAssignee (if exists)
    if not bd_assignee_pivot.empty:
        bd_assignee_pivot.to_excel(writer, index=False, sheet_name="CurrentByAssignee")

    # 3) Charge_Person
    daily_charge_by_person.to_excel(writer, sheet_name="Charge_Person", index=False)

    # 4) Charge_Task
    daily_charge_by_task.to_excel(writer, sheet_name="Charge_Task", index=False)

    # 5) CurrentSprint (if exists)
    if not bd_current.empty:
        bd_current.to_excel(writer, index=False, sheet_name="CurrentSprint")

    # 6) PreviousSprint (if exists)
    if not bd_prev.empty:
        bd_prev.to_excel(writer, index=False, sheet_name="PreviousSprint")

    # --- Workbook & base sheets ---
    wb = writer.book
    ws_status = writer.sheets["Status"]
    ws_by_person = writer.sheets["Charge_Person"]
    ws_by_task = writer.sheets["Charge_Task"]

    # ---------- Common formats ----------
    number_fmt = wb.add_format({'num_format': '#,##0.0;-#,##0.0;"-"'})
    percent_fmt = wb.add_format({"num_format": "0.0%"})
    header_fmt = wb.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'align': 'center',
        'border': 1
    })
    header_fmt_blue = wb.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'align': 'center',
        'border': 1,
        'font_color': '#0070C0'
    })
    blue_number_fmt = wb.add_format({
        'num_format': '#,##0.0;-#,##0.0;"-"',
        'font_color': '#0070C0'
    })
    header_horizontal = wb.add_format({
        "bold": True, "align": "center", "valign": "top",
        "text_wrap": False, "border": 1
    })
    header_vertical = wb.add_format({
        "bold": True,
        "align": "center",
        "valign": "bottom",
        "rotation": 90,
        "text_wrap": False,
        "border": 1
    })
    weekend_fill = wb.add_format({
        "bg_color": "#EAB8D8",
        "align": "center",
        "valign": "bottom",
        "rotation": 90,
        "text_wrap": False,
        "border": 1
    })
    weekend_number_fmt = wb.add_format({
        "num_format": '#,##0.0;-#,##0.0;"-"',
        "bg_color": "#EAB8D8",
        "align": "center",
        "valign": "vcenter",
    })

    # ========== STATUS sheet formatting ==========
    for col_idx, col_name in enumerate(summary.columns):
        width = 15
        fmt = None

        if col_name == "IterationName":
            width = 22
        elif col_name == "Epic":
            width = 22
        elif col_name == "User Story":
            width = 40
        elif col_name == "Title":
            width = 60
        elif col_name == "Priority":
            width = 7
        elif col_name in ("AssignedTo", "State"):
            width = 18
        elif col_name in ("OriginalEstimate", "CompletedWork", "RemainingWork"):
            width = 15
            fmt = number_fmt
        elif col_name in completed_cols:
            width = 15
            fmt = blue_number_fmt
        elif col_name == "WorkItemURL":
            width = 60
        elif col_name in ("IterationStart", "IterationEnd"):
            width = 18

        ws_status.set_column(col_idx, col_idx, width, fmt)

    for col_num, header in enumerate(summary.columns):
        fmt = header_fmt_blue if header in completed_cols else header_fmt
        ws_status.write(0, col_num, header.replace(" ", "\n"), fmt)
    ws_status.set_row(0, 47)
    ws_status.freeze_panes(1, 0)

    if "State" in summary.columns:
        col_state = summary.columns.get_loc("State")
        col_letter = xl_col_to_name(col_state)
        blocked_fmt = wb.add_format({'bg_color': '#FFCDCE'})

        first_row = 1
        last_row = len(summary)
        first_col = 0
        last_col = len(summary.columns) - 1

        ws_status.conditional_format(
            first_row, first_col, last_row, last_col,
            {
                'type': 'formula',
                'criteria': f'=${col_letter}{first_row+1}="Blocked"',
                'format': blocked_fmt
            }
        )

    # ========== BY PERSON sheet formatting ==========
    ws_by_person.set_column(0, 0, 22)  # Assigned To
    ws_by_person.set_column(1, len(daily_charge_by_person.columns) - 2, 4, number_fmt)
    ws_by_person.set_column(
        len(daily_charge_by_person.columns) - 1,
        len(daily_charge_by_person.columns) - 1,
        12,
        number_fmt
    )

    date_cols_person = [
        c for c in daily_charge_by_person.columns
        if c not in ("Assigned To", "total_charge") and isinstance(c, str) and c[:4].isdigit()
    ]

    for col_idx, col_name in enumerate(daily_charge_by_person.columns):
        is_date_col = col_name in date_cols_person
        fmt_header = header_vertical if is_date_col else header_horizontal
        fmt_data = number_fmt if is_date_col else None

        if is_date_col:
            try:
                dt = pd.to_datetime(col_name).date()
                if dt.weekday() >= 5:
                    fmt_header = weekend_fill
                    fmt_data = weekend_number_fmt
            except Exception:
                pass

        ws_by_person.write(0, col_idx, col_name, fmt_header)

        if is_date_col:
            for row_idx, cell_value in enumerate(daily_charge_by_person[col_name], start=1):
                ws_by_person.write(row_idx, col_idx, cell_value, fmt_data)

    ws_by_person.set_row(0, 61)
    ws_by_person.freeze_panes(1, 1)

    # ========== BY TASK sheet formatting ==========
    ws_by_task.set_column(0, 1, 22)
    ws_by_task.set_column(2, 2, 15)
    ws_by_task.set_column(3, 3, 40)
    ws_by_task.set_column(4, len(daily_charge_by_task.columns) - 3, 4, number_fmt)
    ws_by_task.set_column(
        len(daily_charge_by_task.columns) - 2,
        len(daily_charge_by_task.columns) - 1,
        12,
        number_fmt
    )

    date_cols_task = [
        c for c in daily_charge_by_task.columns
        if c not in ("Assigned To", "IterationName", "WorkItemType", "Title", "total_charge", "WorkItemURL")
        and isinstance(c, str) and c[:4].isdigit()
    ]

    for col_idx, col_name in enumerate(daily_charge_by_task.columns):
        is_date_col = col_name in date_cols_task
        fmt_header = header_vertical if is_date_col else header_horizontal
        fmt_data = number_fmt if is_date_col else None

        if is_date_col:
            try:
                dt = pd.to_datetime(col_name).date()
                if dt.weekday() >= 5:
                    fmt_header = weekend_fill
                    fmt_data = weekend_number_fmt
            except Exception:
                pass

        ws_by_task.write(0, col_idx, col_name, fmt_header)

        if is_date_col:
            for row_idx, cell_value in enumerate(daily_charge_by_task[col_name], start=1):
                ws_by_task.write(row_idx, col_idx, cell_value, fmt_data)

    ws_by_task.set_row(0, 61)
    ws_by_task.freeze_panes(1, 4)

    # ========== BURNDOWN SHEETS & CHARTS ==========

    # 1) Current sprint burndown
    if not bd_current.empty:
        ws_curr = writer.sheets["CurrentSprint"]

        for col_idx, col_name in enumerate(bd_current.columns):
            ws_curr.write(0, col_idx, col_name, header_fmt)
        ws_curr.freeze_panes(1, 0)

        chart_curr = wb.add_chart({"type": "line"})
        max_row_curr = len(bd_current)
        chart_curr.add_series({
            "name":       f"Burndown - {current_sprint}",
            "categories": ["CurrentSprint", 1, 0, max_row_curr, 0],
            "values":     ["CurrentSprint", 1, 1, max_row_curr, 1],
        })
        chart_curr.set_title({"name": f"Burndown - {current_sprint}"})
        chart_curr.set_x_axis({"name": "Date"})
        chart_curr.set_y_axis({"name": "Remaining Work (hours)"})
        ws_curr.insert_chart("D2", chart_curr)

    # 2) Previous sprint burndown
    if not bd_prev.empty:
        ws_prev = writer.sheets["PreviousSprint"]

        for col_idx, col_name in enumerate(bd_prev.columns):
            ws_prev.write(0, col_idx, col_name, header_fmt)
        ws_prev.freeze_panes(1, 0)

        chart_prev = wb.add_chart({"type": "line"})
        max_row_prev = len(bd_prev)
        chart_prev.add_series({
            "name":       f"Burndown - {previous_sprint}",
            "categories": ["PreviousSprint", 1, 0, max_row_prev, 0],
            "values":     ["PreviousSprint", 1, 1, max_row_prev, 1],
        })
        chart_prev.set_title({"name": f"Burndown - {previous_sprint}"})
        chart_prev.set_x_axis({"name": "Date"})
        chart_prev.set_y_axis({"name": "Remaining Work (hours)"})
        ws_prev.insert_chart("D2", chart_prev)

    # 3) Current sprint by assignee: line + colored histogram
    if not bd_assignee_pivot.empty:
        ws_ass = writer.sheets["CurrentByAssignee"]

        for col_idx, col_name in enumerate(bd_assignee_pivot.columns):
            ws_ass.write(0, col_idx, col_name, header_fmt)
        ws_ass.freeze_panes(1, 1)

        max_row_ass = len(bd_assignee_pivot)

        # Line chart per assignee
        chart_ass = wb.add_chart({"type": "line"})
        for col_idx, col_name in enumerate(bd_assignee_pivot.columns[1:], start=1):
            chart_ass.add_series({
                "name":       ["CurrentByAssignee", 0, col_idx],
                "categories": ["CurrentByAssignee", 1, 0, max_row_ass, 0],  # Date
                "values":     ["CurrentByAssignee", 1, col_idx, max_row_ass, col_idx],
            })

        chart_ass.set_title({"name": f"Burndown by Assignee - {current_sprint}"})
        chart_ass.set_x_axis({"name": "Date"})
        chart_ass.set_y_axis({"name": "Remaining Work (hours)"})
        ws_ass.insert_chart("H2", chart_ass)

        # Completion % row
        progress_row = max_row_ass + 2
        ws_ass.write(progress_row, 0, "Completion %", header_fmt)

        for col_idx, col_name in enumerate(bd_assignee_pivot.columns[1:], start=1):
            start_val = bd_assignee_pivot.iloc[0, col_idx]
            current_val = bd_assignee_pivot.iloc[-1, col_idx]

            if pd.notna(start_val) and start_val != 0:
                pct = (start_val - current_val) / start_val
                pct = max(0.0, min(1.0, pct))
            else:
                pct = 0.0

            ws_ass.write_number(progress_row, col_idx, pct, percent_fmt)

        # Colored histogram
        chart_hist = wb.add_chart({"type": "column"})
        num_assignees = len(bd_assignee_pivot.columns) - 1

        colors = [
            "#4472C4",  # blue
            "#ED7D31",  # orange
            "#A5A5A5",  # gray
            "#FFC000",  # yellow
            "#5B9BD5",  # light blue
            "#70AD47",  # green
        ]
        points = [
            {"fill": {"color": colors[i % len(colors)]}}
            for i in range(num_assignees)
        ]

        chart_hist.add_series({
            "name":       "Completion %",
            "categories": ["CurrentByAssignee", 0, 1, 0, num_assignees],
            "values":     ["CurrentByAssignee", progress_row, 1, progress_row, num_assignees],
            "points":     points,
        })

        chart_hist.set_title({"name": f"Assignee Completion - {current_sprint}"})
        chart_hist.set_y_axis({
            "name": "Completion",
            "min": 0,
            "max": 1,
            "major_unit": 0.25,
            "num_format": "0%",
        })

        ws_ass.insert_chart("H20", chart_hist)

print(f"✅ Done! File saved: {output_file}")


✅ Done! File saved: ../data/MarketRMS_ProjectStatus_2026-04-01_ADO_API.xlsx


## 2. Utilisation calculation

In [18]:
# Prepare data for calculation

sprints = sprints[
    (~sprints['IterationStart'].isna()) &
    (~sprints['IterationEnd'].isna())
]
sprints['IterationStart'] = pd.to_datetime(sprints["IterationStart"])
sprints['IterationEnd'] = pd.to_datetime(sprints["IterationEnd"])

# Filter source whole dataset for utilisation calculation
utilisation = wi_raw_snapshots[["SnapshotDate", "IterationName", "IterationStart", "IterationEnd", "WorkItemType", "Title", "State", "AssignedTo", "CompletedWork", "OriginalEstimate", "RemainingWork", "WorkItemURL"]].copy()
utilisation['IterationStart'] = pd.to_datetime(utilisation["IterationStart"])
utilisation['IterationEnd'] = pd.to_datetime(utilisation["IterationEnd"])
utilisation = utilisation[utilisation["IterationStart"] >= pd.to_datetime(start_date)]

# Select Snapshots at the first day of each Sprint (it should be day's night, so Snapshot is after planning)
utilisation = utilisation[utilisation["SnapshotDate"] == utilisation["IterationStart"]]

# Calculate CompletedWork at the start of sprint
utilisation["PrevDate"] = utilisation["SnapshotDate"] - pd.Timedelta(days=1)
utilisation = utilisation.merge(
    wi_raw_snapshots[["SnapshotDate", "WorkItemURL", "CompletedWork"]],
    left_on=["WorkItemURL", "PrevDate"],
    right_on=["WorkItemURL", "SnapshotDate"],
    how="left",
    suffixes=('', '_start')
).drop(columns = ["SnapshotDate_start", "PrevDate"])
utilisation["Completed_at_sprint_start"] = utilisation["CompletedWork_start"].fillna(0)
utilisation = utilisation.drop(columns = ["CompletedWork", "CompletedWork_start"])

# Calculate Planned_for_sprint
utilisation["Planned_for_sprint"] = (utilisation["OriginalEstimate"] - utilisation["Completed_at_sprint_start"]).clip(lower=0)

### 2-1 Charged time by sprints

In [19]:
# Charged time by sprints

# 1 Melt table with charges
id_cols = ["Assigned To", "IterationName", "WorkItemType", "Title", "WorkItemURL"]
date_cols = [c for c in daily_charge_by_task.columns if c not in id_cols + ["total_charge"]]
charge_by_sprint = daily_charge_by_task.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="SnapshotDate",
    value_name="charge"
)

charge_by_sprint["SnapshotDate"] = pd.to_datetime(charge_by_sprint["SnapshotDate"], errors="coerce")
charge_by_sprint = charge_by_sprint[charge_by_sprint["charge"].fillna(0) != 0]


# Add 'IterationStart', 'IterationEnd' to Iteration name
charge_by_sprint = charge_by_sprint.merge(
    sprints[['IterationName', 'IterationStart', 'IterationEnd']],
    how='left',
    left_on=None,
    right_on=None,
    suffixes=('', '_s')
)

# 2 Match Iteration for each charged date - WorkingIteration
intervals = pd.IntervalIndex.from_arrays(sprints['IterationStart'], sprints['IterationEnd'], closed='both')
idx = intervals.get_indexer(charge_by_sprint['SnapshotDate'])
charge_by_sprint['WorkingIteration'] = np.where(idx >= 0, sprints['IterationName'].to_numpy()[idx], pd.NA)

# 3 Group daily disagregation to Iteration disagregation
charge_by_sprint = (
    charge_by_sprint.groupby(['WorkingIteration', 'Assigned To', 'WorkItemType', 'Title', 'WorkItemURL'], as_index=False)['charge']
        .sum()
)
charge_by_sprint = charge_by_sprint.dropna(subset=['WorkingIteration'])

### 2-2 For each spring - split planned time and unplanned time

In [20]:
# Merge planned time charges to each sprtint tasks 

utilisation = utilisation.merge(
    charge_by_sprint[["WorkItemURL", "WorkingIteration", "charge"]],
    left_on = ["WorkItemURL", "IterationName"],
    right_on = ["WorkItemURL", "WorkingIteration"],
    how = "left"
)
utilisation["charge"] = utilisation["charge"].fillna(0)

# Mark them as planned
utilisation["Planned"] = "yes"

# Formatting
utilisation = utilisation[["IterationName", "AssignedTo", "WorkItemType", "Title", "Planned", "Planned_for_sprint", "charge", "WorkItemURL"]]


In [21]:
# Filter charges from 'charge_by_sprint', which were not planned for each sprint (utilisation["WorkItemURL"] does not match with charge_by_sprint["WorkingIteration"]) 
not_planned_charge = charge_by_sprint.merge(
    utilisation[["WorkItemURL", "IterationName"]],
    left_on = ["WorkItemURL", "WorkingIteration"],
    right_on = ["WorkItemURL", "IterationName"],
    how = "left"
)
not_planned_charge = not_planned_charge[not_planned_charge["IterationName"].isna()]

# Mark these as not planned
not_planned_charge["Planned"] = "no"

# Formatting
not_planned_charge["Planned_for_sprint"] = 0
not_planned_charge = not_planned_charge[["WorkingIteration", "Assigned To", "WorkItemType", "Title", "Planned_for_sprint", "charge", "WorkItemURL", "Planned"]].rename(columns = {"Assigned To": "AssignedTo", "WorkingIteration": "IterationName"})

# Combine planned tasks with not planned tasks - to single dataframe
utilisation = pd.concat([utilisation, not_planned_charge])

# Formatting
utilisation["Completed time"] = utilisation["charge"]


### 2-3 Calculate metrics

In [22]:
# Remove QA from this analysis
utilisation = utilisation[utilisation["AssignedTo"] != "Horin Serhii"]

# Select completed as planned to separate column
utilisation["Completed for planned tasks"] = np.where(
    utilisation["Planned"] == "yes",
    utilisation[["Completed time", "Planned_for_sprint"]].min(axis=1),
    0
)

# Account for underestimation
utilisation["Overcharged_planned_tasks"] = np.where(
    utilisation["Planned"] == "yes",
    (utilisation["Completed time"] - utilisation["Planned_for_sprint"]).clip(lower=0),
    0
)

# Calculate what time team failed to complete within planned sprint
utilisation["NOT DONE but planed"] = (utilisation["Planned_for_sprint"] - utilisation["Completed time"]).clip(lower=0)


# Select other completed time, which was not planned to be part of the sprint
utilisation["Completed non-sprint tasks"] = np.where(
    utilisation["Planned"] == "no",
    utilisation["Completed time"],
    0
)


In [23]:
utilisation_summary = (
    utilisation.groupby(['AssignedTo', 'IterationName'], as_index=False)
    [['Planned_for_sprint', 'Completed for planned tasks', 'NOT DONE but planed', 'Completed time', 'Completed non-sprint tasks', 'Overcharged_planned_tasks']].sum()
)

utilisation_time_map = {
    'Vasilov Denys':  60,        # 80 hours * 70%
    'Vasilov Denys S.': 60,      # 80 hours * 70%
    'Sukonnik Illia V.': 50,     # 80 hours * 60%
    'Kolodieiev Ivan': 50,       # 80 hours * 60%
    'Andrianov Vladyslav': 40,   # 80 hours * 50%
}

utilisation_summary['Utilisation time'] = (
    utilisation_summary['AssignedTo']
        .map(utilisation_time_map)
        .fillna(0)          # or raise / keep NaN
        .astype(int)
)

utilisation_summary = utilisation_summary.sort_values(["AssignedTo", "IterationName"], ascending=False)

utilisation_summary["Underutilised"] = utilisation_summary['Utilisation time'] - utilisation_summary["Completed time"]
utilisation_summary["Overplanned / Underplanned"] = utilisation_summary['Planned_for_sprint'] - utilisation_summary["Utilisation time"]
utilisation_summary["Work hours debt"] = np.maximum(np.minimum(utilisation_summary["Planned_for_sprint"], utilisation_summary["Utilisation time"]), utilisation_summary["Utilisation time"]) - utilisation_summary["Completed time"]
utilisation_summary["Comment on Plan Focus"] = ""
utilisation_summary["Comment on Utilisation"] = ""

utilisation_summary = utilisation_summary[["AssignedTo", "IterationName", "Utilisation time", "Planned_for_sprint", "Overplanned / Underplanned", "Completed for planned tasks", 'Overcharged_planned_tasks', "Completed non-sprint tasks", "Work hours debt", "Comment on Plan Focus"]]
# "Completed time", "Underutilised", "Comment on Utilisation",
# "NOT DONE but planed"

In [24]:
with pd.ExcelWriter(output_file_2, engine="xlsxwriter") as writer:
    # --- write sheets ---
    utilisation_summary.to_excel(writer, sheet_name="Summary", index=False)
    utilisation.to_excel(writer, sheet_name="Details", index=False)

    workbook  = writer.book

    header_fmt = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "align": "center",
        "border": 1,
    })

    def format_sheet(df: pd.DataFrame, sheet_name: str, header_format, extra_pad: int = 2,
                     min_width: int = 10, max_width: int = 60):
        ws = writer.sheets[sheet_name]

        # Apply header format
        for col_idx, col_name in enumerate(df.columns):
            ws.write(0, col_idx, col_name, header_format)

        # Set column widths (based on max of header length and sampled cell lengths)
        for col_idx, col_name in enumerate(df.columns):
            # header length
            header_len = len(str(col_name))

            # content length (sample a bit for speed; adjust if you want full scan)
            if len(df) > 0:
                sample = df.iloc[:500, col_idx].astype(str)  # sample first 500 rows
                content_len = sample.map(len).max()
            else:
                content_len = 0

            width = max(header_len, content_len) + extra_pad
            width = max(min_width, min(width, max_width))

            ws.set_column(col_idx, col_idx, width)

        # Freeze header row + enable autofilter
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, 0, len(df.columns) - 1)

    format_sheet(utilisation_summary, "Summary", header_fmt)
    format_sheet(utilisation, "Details", header_fmt)

output_file_2

PosixPath('../data/MarketRMS_Utilisation_2026-04-01.xlsx')